# Minimal reproducer: `vmap` + `torch.compile(backend='inductor')` correctness bug

Shows that for Z2-symmetric fermionic PEPS with **odd block dimensions >= 5** (D=10,14,18,...),
`torch.compile` after `torch.export` + `torch.vmap` produces **wrong results**.

Even block dims and small odd block dims (D<=6) are unaffected.

**Setup:** exact contraction (no SVD/QR truncation), so the bug is purely in inductor kernel generation.

In [2]:
import quimb as qu
import quimb.tensor as qtn
import symmray as sr
import torch

torch.set_default_device("cuda")
torch.set_default_dtype(torch.float64)
torch.manual_seed(42)

In [3]:
Lx, Ly = 6, 4
nsites = Lx * Ly
B = 2  # batch size

## Define amplitude function and model wrapper

In [4]:
def amplitude(x, tn):
    """Single-sample exact contraction of fermionic PEPS."""
    tnx = tn.isel(
        {tn.site_ind(site): x[i] for i, site in enumerate(tn.sites)}
    )
    # exact contraction — no SVD/QR truncation
    return tnx.contract()

In [5]:
class TNAmplitudeModel(torch.nn.Module):
    """Wraps a TN amplitude function as an nn.Module for export+compile."""

    def __init__(self, fn, tn):
        super().__init__()
        params, skeleton = qtn.pack(tn)
        params_flat, params_pytree = qu.utils.tree_flatten(
            params, get_ref=True
        )
        self.params = torch.nn.ParameterList(
            [torch.as_tensor(x, dtype=torch.float64) for x in params_flat]
        )
        self._skeleton = skeleton
        self._params_pytree = params_pytree
        self._fn = fn

    def _amplitude(self, x, *flat_params):
        params = qu.utils.tree_unflatten(list(flat_params), self._params_pytree)
        tn = qtn.unpack(params, self._skeleton)
        return self._fn(x, tn)

    def forward(self, x):
        return self._amplitude(x, *self.params)

## Generate random configs

In [6]:
N_f = nsites-2  # number of fermions

def random_initial_config(N_f, N_sites, seed=None):
    if seed is not None:
        torch.manual_seed(seed)
    half_filled_config = torch.tensor(
        [1, 2] * (N_sites // 2)
    )
    # Set first (Lx*Ly - N_f) sites to be empty (0)
    empty_sites = list(range(N_sites - N_f))
    doped_config = half_filled_config.clone()
    doped_config[empty_sites] = 0
    # Randomly permute the doped_config
    perm = torch.randperm(N_sites)
    doped_config = doped_config[perm]
    num_1 = torch.sum(doped_config == 1).item()
    num_2 = torch.sum(doped_config == 2).item()
    assert num_1 == N_f // 2 and num_2 == N_f // 2, f"Number of spin up and spin down fermions should be {N_f // 2}, but got {num_1} and {num_2}"

    return doped_config

z2_map = {0:0, 1:2, 2:3, 3:1}
xs = [
    random_initial_config(N_f, nsites, seed=s)
    for s in range(B)
]
# for x in xs:
#     x.copy_(torch.tensor([z2_map[int(xi)] for xi in x], device="cuda"))
xs = torch.stack(xs)
print(f"Configs shape: {xs.shape}")
xs

Configs shape: torch.Size([2, 24])


tensor([[2, 2, 2, 2, 2, 0, 2, 2, 1, 2, 1, 1, 1, 1, 1, 2, 2, 2, 1, 0, 1, 1, 1, 1],
        [0, 1, 1, 2, 2, 0, 2, 1, 1, 1, 2, 2, 1, 1, 1, 2, 2, 1, 2, 2, 1, 2, 2, 1]],
       device='cuda:0')

## Test function: compare eager vs export+compile

In [7]:
def test_D(D):
    """Test eager vs export+vmap+compile for a given bond dim D."""
    print(f"\n{'=' * 60}")
    print(f"D={D}, Z2 block dim={D // 2}, exact contraction, {Lx}x{Ly}")
    print(f"{'=' * 60}")

    # random Z2-symmetric fermionic PEPS
    peps = sr.networks.PEPS_fermionic_rand(
        "Z2", Lx, Ly, D,
        # phys_dim=4,
        phys_dim=[
                (0, 0),
                (1, 0),
                (1, 1),
                (0, 1),
        ],
        subsizes="equal",
        flat=True,
        seed=42,
        dtype="float64",
    )
    # for ts in peps.tensors:
    #     ts_data = ts.data
    #     ts_data.indices[-1]._linearmap = None
    #     ts.modify(data=ts_data)

    # --- Eager baseline ---
    model = TNAmplitudeModel(amplitude, peps)
    params_list = list(model.params)

    # --- torch.export ---
    from torch.export import export

    class _AmpModule(torch.nn.Module):
        def __init__(self, amp_fn):
            super().__init__()
            self._fn = amp_fn

        def forward(self, x, *flat_params):
            return self._fn(x, *flat_params)

    with torch.no_grad():
        exported = export(
            _AmpModule(model._amplitude),
            (xs[0], *params_list),
        )
    exported_module = exported.module()

    # --- torch.vmap ---
    n_params = len(params_list)
    vmapped = torch.vmap(
        exported_module,
        in_dims=(0, *([None] * n_params)),
    )

    with torch.inference_mode():
        vmap_eager = vmapped(xs, *params_list)
        print(f"  Vmap eager:  {vmap_eager}")

    # --- torch.compile ---
    torch._dynamo.reset()
    compiled = torch.compile(vmapped, backend="inductor")

    with torch.inference_mode():
        compiled_results = compiled(xs, *params_list)
        torch.cuda.synchronize()

    diff = (vmap_eager - compiled_results).abs().max().item() / vmap_eager.abs().max().item()
    ok = diff < 1e-6
    print(f"  Compiled vs eager:        {diff:.2e}  {'PASS' if ok else 'FAIL'}")
    print(f"    eager:    {vmap_eager}")
    print(f"    compiled: {compiled_results}")

    return ok, diff

## Run tests across bond dimensions

In [31]:
D_list = [8,10,12,14,16,18,20]#  8, 10
results = {}
for D in D_list:
    results[D] = test_D(D)


D=8, Z2 block dim=4, exact contraction, 6x4
  Vmap eager:  tensor([ 5.5090e+09, -9.3904e+13], device='cuda:0')


W0310 16:21:20.974000 2824905 torch/_inductor/cudagraph_utils.py:207] [0/0] [__cudagraphs] skipping cudagraphs due to skipping cudagraphs due to cpu device (arg25_1). Found from : 
W0310 16:21:20.974000 2824905 torch/_inductor/cudagraph_utils.py:207] [0/0] [__cudagraphs]    File "/home/sijingdu/TNVMC/VMC_code/pytorch_dev/lib/python3.12/site-packages/torch/_functorch/apis.py", line 220, in wrapped
W0310 16:21:20.974000 2824905 torch/_inductor/cudagraph_utils.py:207] [0/0] [__cudagraphs]     return vmap_impl(
W0310 16:21:20.974000 2824905 torch/_inductor/cudagraph_utils.py:207] [0/0] [__cudagraphs]   File "/home/sijingdu/TNVMC/VMC_code/pytorch_dev/lib/python3.12/site-packages/torch/_functorch/vmap.py", line 316, in vmap_impl
W0310 16:21:20.974000 2824905 torch/_inductor/cudagraph_utils.py:207] [0/0] [__cudagraphs]     return _flat_vmap(
W0310 16:21:20.974000 2824905 torch/_inductor/cudagraph_utils.py:207] [0/0] [__cudagraphs]   File "/home/sijingdu/TNVMC/VMC_code/pytorch_dev/lib/python3.

  Compiled vs eager:        0.00e+00  PASS
    eager:    tensor([ 5.5090e+09, -9.3904e+13], device='cuda:0')
    compiled: tensor([ 5.5090e+09, -9.3904e+13], device='cuda:0')

D=10, Z2 block dim=5, exact contraction, 6x4
  Vmap eager:  tensor([-1.2778e+15, -5.5105e+15], device='cuda:0')


W0310 16:22:00.406000 2824905 torch/_inductor/cudagraph_utils.py:207] [0/0] [__cudagraphs] skipping cudagraphs due to skipping cudagraphs due to cpu device (arg25_1). Found from : 
W0310 16:22:00.406000 2824905 torch/_inductor/cudagraph_utils.py:207] [0/0] [__cudagraphs]    File "/home/sijingdu/TNVMC/VMC_code/pytorch_dev/lib/python3.12/site-packages/torch/_functorch/apis.py", line 220, in wrapped
W0310 16:22:00.406000 2824905 torch/_inductor/cudagraph_utils.py:207] [0/0] [__cudagraphs]     return vmap_impl(
W0310 16:22:00.406000 2824905 torch/_inductor/cudagraph_utils.py:207] [0/0] [__cudagraphs]   File "/home/sijingdu/TNVMC/VMC_code/pytorch_dev/lib/python3.12/site-packages/torch/_functorch/vmap.py", line 316, in vmap_impl
W0310 16:22:00.406000 2824905 torch/_inductor/cudagraph_utils.py:207] [0/0] [__cudagraphs]     return _flat_vmap(
W0310 16:22:00.406000 2824905 torch/_inductor/cudagraph_utils.py:207] [0/0] [__cudagraphs]   File "/home/sijingdu/TNVMC/VMC_code/pytorch_dev/lib/python3.

  Compiled vs eager:        0.00e+00  PASS
    eager:    tensor([-1.2778e+15, -5.5105e+15], device='cuda:0')
    compiled: tensor([-1.2778e+15, -5.5105e+15], device='cuda:0')

D=12, Z2 block dim=6, exact contraction, 6x4
  Vmap eager:  tensor([6.7217e+16, 1.5389e+16], device='cuda:0')


...

  Compiled vs eager:        0.00e+00  PASS
    eager:    tensor([-3.7397e+19,  6.0568e+19], device='cuda:0')
    compiled: tensor([-3.7397e+19,  6.0568e+19], device='cuda:0')

D=18, Z2 block dim=9, exact contraction, 6x4
  Vmap eager:  tensor([3.8539e+20, 1.6763e+20], device='cuda:0')


W0310 16:24:55.354000 2824905 torch/_inductor/cudagraph_utils.py:207] [0/0] [__cudagraphs] skipping cudagraphs due to skipping cudagraphs due to cpu device (arg25_1). Found from : 
W0310 16:24:55.354000 2824905 torch/_inductor/cudagraph_utils.py:207] [0/0] [__cudagraphs]    File "/home/sijingdu/TNVMC/VMC_code/pytorch_dev/lib/python3.12/site-packages/torch/_functorch/apis.py", line 220, in wrapped
W0310 16:24:55.354000 2824905 torch/_inductor/cudagraph_utils.py:207] [0/0] [__cudagraphs]     return vmap_impl(
W0310 16:24:55.354000 2824905 torch/_inductor/cudagraph_utils.py:207] [0/0] [__cudagraphs]   File "/home/sijingdu/TNVMC/VMC_code/pytorch_dev/lib/python3.12/site-packages/torch/_functorch/vmap.py", line 316, in vmap_impl
W0310 16:24:55.354000 2824905 torch/_inductor/cudagraph_utils.py:207] [0/0] [__cudagraphs]     return _flat_vmap(
W0310 16:24:55.354000 2824905 torch/_inductor/cudagraph_utils.py:207] [0/0] [__cudagraphs]   File "/home/sijingdu/TNVMC/VMC_code/pytorch_dev/lib/python3.

  Compiled vs eager:        0.00e+00  PASS
    eager:    tensor([3.8539e+20, 1.6763e+20], device='cuda:0')
    compiled: tensor([3.8539e+20, 1.6763e+20], device='cuda:0')

D=20, Z2 block dim=10, exact contraction, 6x4
  Vmap eager:  tensor([ 3.0953e+21, -4.0432e+20], device='cuda:0')


W0310 16:25:37.926000 2824905 torch/_inductor/cudagraph_utils.py:207] [0/0] [__cudagraphs] skipping cudagraphs due to skipping cudagraphs due to cpu device (arg25_1). Found from : 
W0310 16:25:37.926000 2824905 torch/_inductor/cudagraph_utils.py:207] [0/0] [__cudagraphs]    File "/home/sijingdu/TNVMC/VMC_code/pytorch_dev/lib/python3.12/site-packages/torch/_functorch/apis.py", line 220, in wrapped
W0310 16:25:37.926000 2824905 torch/_inductor/cudagraph_utils.py:207] [0/0] [__cudagraphs]     return vmap_impl(
W0310 16:25:37.926000 2824905 torch/_inductor/cudagraph_utils.py:207] [0/0] [__cudagraphs]   File "/home/sijingdu/TNVMC/VMC_code/pytorch_dev/lib/python3.12/site-packages/torch/_functorch/vmap.py", line 316, in vmap_impl
W0310 16:25:37.926000 2824905 torch/_inductor/cudagraph_utils.py:207] [0/0] [__cudagraphs]     return _flat_vmap(
W0310 16:25:37.926000 2824905 torch/_inductor/cudagraph_utils.py:207] [0/0] [__cudagraphs]   File "/home/sijingdu/TNVMC/VMC_code/pytorch_dev/lib/python3.

  Compiled vs eager:        0.00e+00  PASS
    eager:    tensor([ 3.0953e+21, -4.0432e+20], device='cuda:0')
    compiled: tensor([ 3.0953e+21, -4.0432e+20], device='cuda:0')


In [32]:
print(f"\n{'=' * 60}")
print("Compare eager forward amps and compiled forward amps")
print(f"({Lx}x{Ly} square lattice, exact contraction, cudagraphs)")
print(f"{'=' * 60}")
print(f"  {'D':>4}  {'Z2 block dim':>12}  {'eager==compile?':>16}  {'max rel diff':>10}")
print(f"  {'-'*4}  {'-'*12}  {'-'*16}  {'-'*10}")
for D in D_list:
    ok, diff = results[D]
    bdim = D // 2
    print(
        f"  {D:>4}  {bdim:>12}  "
        f"{'PASS' if ok else 'FAIL':>16}  {diff:>10.2e}"
    )
print(f"{'=' * 60}")


Compare eager forward amps and compiled forward amps
(6x4 square lattice, exact contraction, cudagraphs)
     D  Z2 block dim   eager==compile?  max rel diff
  ----  ------------  ----------------  ----------
     8             4              PASS    0.00e+00
    10             5              PASS    0.00e+00
    12             6              PASS    0.00e+00
    14             7              PASS    0.00e+00
    16             8              PASS    0.00e+00
    18             9              PASS    0.00e+00
    20            10              PASS    0.00e+00


In [29]:
print(f"\n{'=' * 60}")
print("Compare eager forward amps and compiled forward amps")
print(f"({Lx}x{Ly} square lattice, exact contraction, Torchinductor)")
print(f"{'=' * 60}")
print(f"  {'D':>4}  {'Z2 block dim':>12}  {'eager==compile?':>16}  {'max rel diff':>10}")
print(f"  {'-'*4}  {'-'*12}  {'-'*16}  {'-'*10}")
for D in D_list:
    ok, diff = results[D]
    bdim = D // 2
    print(
        f"  {D:>4}  {bdim:>12}  "
        f"{'PASS' if ok else 'FAIL':>16}  {diff:>10.2e}"
    )
print(f"{'=' * 60}")


Compare eager forward amps and compiled forward amps
(6x4 square lattice, exact contraction, Torchinductor)
     D  Z2 block dim   eager==compile?  max rel diff
  ----  ------------  ----------------  ----------
     8             4              PASS    4.99e-16
    10             5              FAIL    9.70e-01
    12             6              PASS    3.03e-15
    14             7              FAIL    3.72e-01
    16             8              PASS    5.41e-16
    18             9              FAIL    1.89e+00
    20            10              PASS    2.33e-15


## Expected result

| D | Z2 block dim | Status | Notes |
|---|---|---|---|
| 4 | 2 | PASS | |
| 6 | 3 | PASS | |
| 8 | 4 | PASS | |
| **10** | **5** | **FAIL** | odd block dim |
| 12 | 6 | PASS | |
| **14** | **7** | **FAIL** | odd block dim |
| 16 | 8 | PASS | |
| **18** | **9** | **FAIL** | odd block dim |
| 20 | 10 | PASS | |

**Pattern:** Bug hits odd Z2 block dims >= 5. Even block dims and small odd (2, 3) are fine.

Eager vmap is always correct — the bug is purely in the inductor backend.

## Speedup from compilation

Benchmark vmap-eager vs vmap+compile (inductor and cudagraphs) for each D.

In [ ]:
import time

N_warmup = 3
N_repeat = 20

def benchmark_D(D):
    """Benchmark vmap-eager vs inductor vs cudagraphs for a given D."""
    print(f"\n{'=' * 60}")
    print(f"D={D}, Z2 block dim={D // 2}, exact contraction, {Lx}x{Ly}")
    print(f"{'=' * 60}")

    peps = sr.networks.PEPS_fermionic_rand(
        "Z2", Lx, Ly, D,
        phys_dim=[
            (0, 0),
            (1, 0),
            (1, 1),
            (0, 1),
        ],
        subsizes="equal",
        flat=True,
        seed=42,
        dtype="float64",
    )

    model = TNAmplitudeModel(amplitude, peps)
    params_list = list(model.params)

    # --- torch.export + vmap ---
    from torch.export import export

    class _AmpModule(torch.nn.Module):
        def __init__(self, amp_fn):
            super().__init__()
            self._fn = amp_fn
        def forward(self, x, *flat_params):
            return self._fn(x, *flat_params)

    with torch.no_grad():
        exported = export(
            _AmpModule(model._amplitude),
            (xs[0], *params_list),
        )
    exported_module = exported.module()
    n_params = len(params_list)
    vmapped = torch.vmap(
        exported_module,
        in_dims=(0, *([None] * n_params)),
    )

    # --- Benchmark vmap eager ---
    with torch.inference_mode():
        for _ in range(N_warmup):
            vmapped(xs, *params_list)
            torch.cuda.synchronize()
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(N_repeat):
            vmapped(xs, *params_list)
            torch.cuda.synchronize()
        t_eager = (time.perf_counter() - t0) / N_repeat

    # --- Benchmark inductor ---
    torch._dynamo.reset()
    compiled_inductor = torch.compile(vmapped, backend="inductor")
    with torch.inference_mode():
        for _ in range(N_warmup):
            compiled_inductor(xs, *params_list)
            torch.cuda.synchronize()
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(N_repeat):
            compiled_inductor(xs, *params_list)
            torch.cuda.synchronize()
        t_inductor = (time.perf_counter() - t0) / N_repeat

    # --- Benchmark cudagraphs ---
    torch._dynamo.reset()
    compiled_cudagraphs = torch.compile(vmapped, backend="cudagraphs")
    with torch.inference_mode():
        for _ in range(N_warmup):
            compiled_cudagraphs(xs, *params_list)
            torch.cuda.synchronize()
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(N_repeat):
            compiled_cudagraphs(xs, *params_list)
            torch.cuda.synchronize()
        t_cudagraphs = (time.perf_counter() - t0) / N_repeat

    print(f"  vmap eager:   {t_eager*1e3:8.2f} ms")
    print(f"  inductor:     {t_inductor*1e3:8.2f} ms  ({t_eager/t_inductor:.1f}x speedup)")
    print(f"  cudagraphs:   {t_cudagraphs*1e3:8.2f} ms  ({t_eager/t_cudagraphs:.1f}x speedup)")

    return t_eager, t_inductor, t_cudagraphs

In [ ]:
timing = {}
for D in D_list:
    timing[D] = benchmark_D(D)

In [ ]:
print(f"\n{'=' * 72}")
print(f"Compilation speedup summary ({Lx}x{Ly}, exact contraction, B={B})")
print(f"{'=' * 72}")
print(
    f"  {'D':>4}  {'block':>5}  "
    f"{'eager (ms)':>10}  {'inductor (ms)':>13}  {'cudagraphs (ms)':>15}  "
    f"{'ind. speedup':>12}  {'cg. speedup':>11}"
)
print(f"  {'-'*4}  {'-'*5}  {'-'*10}  {'-'*13}  {'-'*15}  {'-'*12}  {'-'*11}")
for D in D_list:
    t_e, t_i, t_c = timing[D]
    print(
        f"  {D:>4}  {D//2:>5}  "
        f"{t_e*1e3:>10.2f}  {t_i*1e3:>13.2f}  {t_c*1e3:>15.2f}  "
        f"{t_e/t_i:>12.1f}x  {t_e/t_c:>10.1f}x"
    )
print(f"{'=' * 72}")